In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

In [2]:
# Paths
repo_root = Path.cwd()
while not (repo_root / "results").exists() and repo_root.parent is not None:
    if (repo_root / "results").exists():
        break
    repo_root = repo_root.parent

RESULTS_DIR = repo_root / "results" / "arch_grid_search_shards"

if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"Results directory not found: {RESULTS_DIR}")

print("Using repo_root:", repo_root)
print("Using results dir:", RESULTS_DIR)

Using repo_root: /ocean/projects/cis260122p/gfeng/lob-deep-survival-analysis
Using results dir: /ocean/projects/cis260122p/gfeng/lob-deep-survival-analysis/results/arch_grid_search_shards


In [3]:
# Discover shard result files
csv_files = sorted(RESULTS_DIR.glob("arch_grid_results_*.csv"))
json_files = sorted(RESULTS_DIR.glob("arch_grid_results_*.json"))

print(f"Found {len(csv_files)} shard CSV files")
print(f"Found {len(json_files)} shard metadata files")

if len(csv_files) == 0:
    raise ValueError(f"No shard CSV files found in {RESULTS_DIR}")

Found 2471 shard CSV files
Found 2471 shard metadata files


In [4]:
# ---- Optional filtering by array_job_id ----

# Set to None to load everything
# Or provide a list like: ["123456", "123457"]
FILTER_JOB_IDS = ['40502562', '40464711']

def extract_job_id_from_filename(path: Path) -> str | None:
    """
    Extract array_job_id from filename pattern:
    arch_grid_results_<config>_<jobid>_<taskid>.csv
    """
    parts = path.stem.split("_")
    if len(parts) < 2:
        return None
    # last two parts are jobid and taskid
    # e.g. ..._<jobid>_<taskid>
    try:
        job_id = parts[-2]
        return job_id
    except Exception:
        return None


filtered_csv_files = []

for path in csv_files:
    job_id = extract_job_id_from_filename(path)

    if FILTER_JOB_IDS is None:
        keep = True
    else:
        keep = job_id in set(str(x) for x in FILTER_JOB_IDS)

    if keep:
        filtered_csv_files.append(path)

print(f"CSV files before filtering: {len(csv_files)}")
print(f"CSV files after filtering: {len(filtered_csv_files)}")

if len(filtered_csv_files) == 0:
    raise ValueError("No CSV files matched FILTER_JOB_IDS")

# overwrite csv_files for downstream cells
csv_files = sorted(filtered_csv_files)

CSV files before filtering: 2471
CSV files after filtering: 1322


In [5]:
def safe_read_csv(path: Path) -> pd.DataFrame | None:
    try:
        df = pd.read_csv(path)
        df["source_csv"] = path.name
        return df
    except Exception as e:
        warnings.warn(f"Failed to read {path.name}: {e}")
        return None

dfs = []
for path in csv_files:
    df = safe_read_csv(path)
    if df is not None and len(df) > 0:
        dfs.append(df)

if not dfs:
    raise ValueError("All shard CSV files failed to load or were empty.")

raw_df = pd.concat(dfs, ignore_index=True)
print("Merged raw rows:", len(raw_df))
print("Columns:", sorted(raw_df.columns.tolist()))

Merged raw rows: 1322
Columns: ['alpha', 'array_job_id', 'array_task_id', 'beta_l3', 'config_id', 'created_at', 'depth_repr', 'fc_dropout', 'fc_hidden_ratio', 'gru_layers', 'hidden_size', 'learning_rate', 'mamba_d_conv', 'mamba_d_state', 'mamba_dropout', 'mamba_expand', 'mamba_layers', 'model_name', 'num_epochs', 'rnn_dropout', 'run_name', 'seed', 'selection_max_params', 'selection_min_params', 'source_csv', 'ticker', 'trainable_params', 'trainable_params_m', 'transformer_dropout', 'transformer_ff_ratio', 'transformer_heads', 'transformer_layers', 'val_ctd_favorable', 'val_ctd_toxic', 'val_ibs_favorable', 'val_ibs_toxic', 'val_ibs_weighted', 'val_ibs_weighted_uninformed', 'val_toxic_pr_auc_sample', 'val_toxic_pr_auc_weighted', 'val_toxic_roc_auc_sample', 'val_toxic_roc_auc_weighted', 'val_weighted_ctd']


In [6]:
# Basic normalization / numeric conversion
metric_cols = [
    "val_weighted_ctd",
    "val_toxic_pr_auc_weighted",
    "val_toxic_pr_auc_sample",
    "val_ctd_toxic",
    "val_ctd_favorable",
    "val_ibs_weighted",
]

for col in metric_cols:
    if col in raw_df.columns:
        raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

# Normalize key ID fields
for col in ["config_id", "model_name", "ticker", "source_csv"]:
    if col in raw_df.columns:
        raw_df[col] = raw_df[col].astype(str)

if "seed" in raw_df.columns:
    raw_df["seed"] = pd.to_numeric(raw_df["seed"], errors="coerce").astype("Int64")

print(raw_df[["config_id", "model_name", "ticker", "seed"] + [c for c in metric_cols if c in raw_df.columns]].head())

      config_id model_name ticker  seed  val_weighted_ctd  \
0  019a5e5e58f5      mamba   AAPL  4718          0.574951   
1  019a5e5e58f5      mamba   AAPL  4819          0.563609   
2  019a5e5e58f5      mamba   AAPL  4920          0.548822   
3  019a5e5e58f5      mamba   INTC  4718          0.656997   
4  019a5e5e58f5      mamba   INTC  4920          0.671535   

   val_toxic_pr_auc_weighted  val_toxic_pr_auc_sample  val_ctd_toxic  \
0                   0.289461                 0.205142       0.685944   
1                   0.262711                 0.204083       0.709040   
2                   0.282162                 0.220256       0.655520   
3                   0.562929                 0.415282       0.703511   
4                   0.575697                 0.426841       0.701956   

   val_ctd_favorable  val_ibs_weighted  
0           0.550151          0.235330  
1           0.531114          0.197407  
2           0.524982          0.191429  
3           0.622605          0.2470

In [7]:
# Error handling for failed shards / incomplete rows:
# Keep rows even if some metrics are missing, but exclude missing metrics from each ranking calculation.

required_id_cols = ["config_id", "model_name", "ticker", "seed"]
missing_id_cols = [c for c in required_id_cols if c not in raw_df.columns]
if missing_id_cols:
    raise ValueError(f"Missing required ID columns: {missing_id_cols}")

# Detect duplicate (config, ticker, seed) rows and keep the best available one
# Preference order:
# 1) row with val_weighted_ctd present
# 2) row with val_toxic_pr_auc_weighted present
# 3) first row otherwise
dedup_df = raw_df.copy()
dedup_df["_has_ctd"] = dedup_df["val_weighted_ctd"].notna() if "val_weighted_ctd" in dedup_df.columns else False
dedup_df["_has_prauc"] = dedup_df["val_toxic_pr_auc_weighted"].notna() if "val_toxic_pr_auc_weighted" in dedup_df.columns else False

dedup_df = dedup_df.sort_values(
    by=["config_id", "ticker", "seed", "_has_ctd", "_has_prauc"],
    ascending=[True, True, True, False, False],
)

dedup_df = dedup_df.drop_duplicates(subset=["config_id", "ticker", "seed"], keep="first").copy()

print("Rows after dedup:", len(dedup_df))
print("Unique configs:", dedup_df["config_id"].nunique())

Rows after dedup: 1322
Unique configs: 148


In [8]:
# Coverage diagnostics
coverage_df = (
    dedup_df.groupby(["config_id", "model_name"], as_index=False)
    .agg(
        n_rows=("config_id", "size"),
        n_unique_tickers=("ticker", "nunique"),
        n_unique_seeds=("seed", "nunique"),
        n_ctd=("val_weighted_ctd", lambda s: s.notna().sum()),
        n_prauc=("val_toxic_pr_auc_weighted", lambda s: s.notna().sum()),
        trainable_params=("trainable_params", "first") if "trainable_params" in dedup_df.columns else ("config_id", "size"),
        hidden_size=("hidden_size", "first") if "hidden_size" in dedup_df.columns else ("config_id", "size"),
    )
    .sort_values(["model_name", "config_id"])
    .reset_index(drop=True)
)

coverage_df["expected_rows"] = 6  # 3 tickers x 2 seeds
coverage_df["missing_rows"] = coverage_df["expected_rows"] - coverage_df["n_rows"]

display(coverage_df.head(20))

,config_id,model_name,n_rows,n_unique_tickers,n_unique_seeds,n_ctd,n_prauc,trainable_params,hidden_size,expected_rows,missing_rows
0,0ac6a244c89a,gru,9,3,3,9,9,485613,160,6,-3
1,3333e78c45c6,gru,9,3,3,9,9,415901,128,6,-3
2,6b133a394a8b,gru,9,3,3,9,9,467901,192,6,-3
3,83caef7ac194,gru,9,3,3,9,9,217757,128,6,-3
4,8bf3464c2bd8,gru,9,3,3,9,9,331053,160,6,-3
5,8e60200ad5d3,gru,9,3,3,9,9,239757,96,6,-3
6,ac0b56427ed0,gru,9,3,3,9,9,316829,128,6,-3
7,06f667656266,gru_transformer,9,3,3,9,9,418701,96,6,-3
8,103eb94557be,gru_transformer,9,3,3,9,9,480285,128,6,-3
9,1e14f24947d2,gru_transformer,9,3,3,9,9,418701,96,6,-3


In [9]:
# Aggregate metrics per config, using available rows only
agg_dict = {
    "val_weighted_ctd": "mean",
    "val_toxic_pr_auc_weighted": "mean",
}

extra_first_cols = [
    "trainable_params",
    "trainable_params_m",
    "hidden_size",
    "gru_layers",
    "transformer_layers",
    "transformer_heads",
    "transformer_ff_ratio",
    "mamba_layers",
    "mamba_d_state",
    "mamba_d_conv",
    "mamba_expand",
    "depth_repr",
]

group_cols = ["config_id", "model_name"]

agg_spec = {
    "avg_weighted_ctd": ("val_weighted_ctd", "mean"),
    "avg_toxic_pr_auc": ("val_toxic_pr_auc_weighted", "mean"),
    "n_ctd": ("val_weighted_ctd", lambda s: s.notna().sum()),
    "n_prauc": ("val_toxic_pr_auc_weighted", lambda s: s.notna().sum()),
    "n_total_rows": ("config_id", "size"),
    "n_unique_tickers": ("ticker", "nunique"),
    "n_unique_seeds": ("seed", "nunique"),
}

for col in extra_first_cols:
    if col in dedup_df.columns:
        agg_spec[col] = (col, "first")

rank_df = (
    dedup_df.groupby(group_cols, as_index=False)
    .agg(**agg_spec)
    .copy()
)

rank_df["expected_rows"] = 6
rank_df["ctd_complete"] = rank_df["n_ctd"] == rank_df["expected_rows"]
rank_df["prauc_complete"] = rank_df["n_prauc"] == rank_df["expected_rows"]

display(rank_df.head(20))

,config_id,model_name,avg_weighted_ctd,avg_toxic_pr_auc,n_ctd,n_prauc,n_total_rows,n_unique_tickers,n_unique_seeds,trainable_params,...,transformer_heads,transformer_ff_ratio,mamba_layers,mamba_d_state,mamba_d_conv,mamba_expand,depth_repr,expected_rows,ctd_complete,prauc_complete
0,019a5e5e58f5,mamba,0.689785,0.447714,8,8,8,3,3,487485,...,NaN,NaN,1.0,8.0,2.0,2.0,mamba=1.0,6,False,False
1,052603f85d52,mamba,0.696554,0.448081,9,9,9,3,3,325581,...,NaN,NaN,4.0,8.0,2.0,2.0,mamba=4.0,6,False,False
2,06f667656266,gru_transformer,0.633123,0.465316,9,9,9,3,3,418701,...,2.0,3.0,NaN,NaN,NaN,NaN,"gru=2.0,tr=2.0",6,False,False
3,07660a4611ce,mamba,0.710925,0.450593,9,9,9,3,3,353773,...,NaN,NaN,1.0,16.0,2.0,2.0,mamba=1.0,6,False,False
4,0ac6a244c89a,gru,0.709324,0.481430,9,9,9,3,3,485613,...,NaN,NaN,NaN,NaN,NaN,NaN,gru=2.0,6,False,False
5,103eb94557be,gru_transformer,0.656952,0.452113,9,9,9,3,3,480285,...,4.0,4.0,NaN,NaN,NaN,NaN,"gru=1.0,tr=1.0",6,False,False
6,11a39cdcdf6e,mamba,0.698533,0.474346,9,9,9,3,3,208845,...,NaN,NaN,2.0,16.0,4.0,2.0,mamba=2.0,6,False,False
7,15192c9873d9,mamba,0.696568,0.467276,9,9,9,3,3,235421,...,NaN,NaN,1.0,16.0,4.0,2.0,mamba=1.0,6,False,False
8,160d7bd240f9,transformer,0.637552,0.447668,9,9,9,3,3,306765,...,2.0,3.0,NaN,NaN,NaN,NaN,tr=2.0,6,False,False
9,1906f0bdfcc7,mamba,0.681369,0.459686,9,9,9,3,3,375709,...,NaN,NaN,2.0,32.0,2.0,2.0,mamba=2.0,6,False,False


In [10]:
# Split into 4 DataFrames by model_name

MODEL_TYPES = ["gru", "gru_transformer", "transformer", "mamba"]

df_by_model = {}

for model_name in MODEL_TYPES:
    df_by_model[model_name] = rank_df[rank_df["model_name"] == model_name].copy()
    print(f"{model_name}: {len(df_by_model[model_name])} configs")

gru: 7 configs
gru_transformer: 39 configs
transformer: 36 configs
mamba: 66 configs


In [16]:
# Define ranking function per model

def rank_per_model(df: pd.DataFrame):
    df = df.copy()

    df["ctd_rank"] = df["avg_weighted_ctd"].rank(
        method="min", ascending=False, na_option="bottom"
    )

    df["prauc_rank"] = df["avg_toxic_pr_auc"].rank(
        method="min", ascending=False, na_option="bottom"
    )

    df["combined_rank_score"] = (df["ctd_rank"] + df["prauc_rank"])

    df_ctd = df.sort_values(
        by=["avg_weighted_ctd", "avg_toxic_pr_auc"],
        ascending=[False, False],
    ).reset_index(drop=True)

    df_prauc = df.sort_values(
        by=["avg_toxic_pr_auc", "avg_weighted_ctd"],
        ascending=[False, False],
    ).reset_index(drop=True)

    df_combined = df.sort_values(
        by=["combined_rank_score", "ctd_rank", "prauc_rank"],
        ascending=[True, True, True],
    ).reset_index(drop=True)

    return df_ctd, df_prauc, df_combined

In [ ]:
# Display top 5 per model for each ranking method

display_cols = [
    "config_id",
    "avg_weighted_ctd",
    "avg_toxic_pr_auc",
    "ctd_rank",
    "prauc_rank",
    "combined_rank_score",
    "n_ctd",
    "n_prauc",
]

optional_cols = [
    "trainable_params",
    "hidden_size",
    "depth_repr",
]

display_cols += [c for c in optional_cols if c in rank_df.columns]

for model_name, df_model in df_by_model.items():
    if df_model.empty:
        continue

    print(f"\n===== {model_name.upper()} =====")

    display_cols_model = display_cols.copy()
    if 'gru' in model_name:
        display_cols_model += [
            "gru_layers",
        ]
    if 'mamba' in model_name:
        display_cols_model += [
            "mamba_layers",
            "mamba_d_state",
            "mamba_d_conv",
            "mamba_expand",
        ]
    if 'transformer' in model_name:
        display_cols_model += [
            "transformer_layers",
            "transformer_heads",
            "transformer_ff_ratio",
        ]

    df_ctd, df_prauc, df_combined = rank_per_model(df_model)

    print("\nTop 5 by avg weighted CTD")
    display(df_ctd[display_cols_model].head(5))

    print("\nTop 5 by avg toxic PR AUC")
    display(df_prauc[display_cols_model].head(5))

    print("\nTop 5 by combined rank score")
    display(df_combined[display_cols_model].head(5))


===== GRU =====

Top 5 by avg weighted CTD


,config_id,avg_weighted_ctd,avg_toxic_pr_auc,ctd_rank,prauc_rank,combined_rank_score,n_ctd,n_prauc,trainable_params,hidden_size,depth_repr,gru_layers
0,8bf3464c2bd8,0.734724,0.437933,1.0,6.0,7.0,9,9,331053,160,gru=1.0,1.0
1,0ac6a244c89a,0.709324,0.481430,2.0,2.0,4.0,9,9,485613,160,gru=2.0,2.0
2,83caef7ac194,0.698237,0.451996,3.0,4.0,7.0,9,9,217757,128,gru=1.0,1.0
3,ac0b56427ed0,0.685034,0.481014,4.0,3.0,7.0,9,9,316829,128,gru=2.0,2.0
4,3333e78c45c6,0.683582,0.445060,5.0,5.0,10.0,9,9,415901,128,gru=3.0,3.0



Top 5 by avg toxic PR AUC


,config_id,avg_weighted_ctd,avg_toxic_pr_auc,ctd_rank,prauc_rank,combined_rank_score,n_ctd,n_prauc,trainable_params,hidden_size,depth_repr,gru_layers
0,8e60200ad5d3,0.668478,0.481648,7.0,1.0,8.0,9,9,239757,96,gru=3.0,3.0
1,0ac6a244c89a,0.709324,0.481430,2.0,2.0,4.0,9,9,485613,160,gru=2.0,2.0
2,ac0b56427ed0,0.685034,0.481014,4.0,3.0,7.0,9,9,316829,128,gru=2.0,2.0
3,83caef7ac194,0.698237,0.451996,3.0,4.0,7.0,9,9,217757,128,gru=1.0,1.0
4,3333e78c45c6,0.683582,0.445060,5.0,5.0,10.0,9,9,415901,128,gru=3.0,3.0



Top 5 by combined rank score


...

,config_id,avg_weighted_ctd,avg_toxic_pr_auc,ctd_rank,prauc_rank,combined_rank_score,n_ctd,n_prauc,trainable_params,hidden_size,depth_repr,mamba_layers,mamba_d_state,mamba_d_conv,mamba_expand
0,3adfed4cdef2,0.733845,0.452868,1.0,37.0,38.0,9,9,234909,128,mamba=1.0,1.0,16.0,2.0,2.0
1,43dc32bbcf37,0.728337,0.483927,2.0,2.0,4.0,8,8,284997,144,mamba=1.0,1.0,8.0,4.0,2.0
2,fe45bac7b665,0.724112,0.460634,3.0,24.0,27.0,9,9,325197,96,mamba=2.0,2.0,8.0,2.0,4.0
3,b54b98129904,0.722210,0.461637,4.0,20.0,24.0,9,9,229277,128,mamba=1.0,1.0,8.0,4.0,2.0
4,94408760b57f,0.720678,0.473091,5.0,7.0,12.0,9,9,291333,144,mamba=1.0,1.0,16.0,2.0,2.0



Top 5 by avg toxic PR AUC


,config_id,avg_weighted_ctd,avg_toxic_pr_auc,ctd_rank,prauc_rank,combined_rank_score,n_ctd,n_prauc,trainable_params,hidden_size,depth_repr,mamba_layers,mamba_d_state,mamba_d_conv,mamba_expand
0,2c95fad373ce,0.687365,0.498257,45.0,1.0,46.0,8,8,464133,144,mamba=2.0,2.0,32.0,2.0,2.0
1,43dc32bbcf37,0.728337,0.483927,2.0,2.0,4.0,8,8,284997,144,mamba=1.0,1.0,8.0,4.0,2.0
2,b8e912be2e3e,0.718306,0.479989,8.0,3.0,11.0,9,9,227277,96,mamba=2.0,2.0,32.0,4.0,2.0
3,883396dabe02,0.651750,0.476126,62.0,4.0,66.0,7,7,463845,144,mamba=1.0,1.0,32.0,2.0,4.0
4,11a39cdcdf6e,0.698533,0.474346,30.0,5.0,35.0,9,9,208845,96,mamba=2.0,2.0,16.0,4.0,2.0



Top 5 by combined rank score


,config_id,avg_weighted_ctd,avg_toxic_pr_auc,ctd_rank,prauc_rank,combined_rank_score,n_ctd,n_prauc,trainable_params,hidden_size,depth_repr,mamba_layers,mamba_d_state,mamba_d_conv,mamba_expand
0,43dc32bbcf37,0.728337,0.483927,2.0,2.0,4.0,8,8,284997,144,mamba=1.0,1.0,8.0,4.0,2.0
1,b8e912be2e3e,0.718306,0.479989,8.0,3.0,11.0,9,9,227277,96,mamba=2.0,2.0,32.0,4.0,2.0
2,94408760b57f,0.720678,0.473091,5.0,7.0,12.0,9,9,291333,144,mamba=1.0,1.0,16.0,2.0,2.0
3,398e1a6bfe57,0.717774,0.471485,10.0,9.0,19.0,9,9,380493,96,mamba=2.0,2.0,32.0,2.0,4.0
4,991570e5f681,0.718043,0.469305,9.0,11.0,20.0,9,9,208077,96,mamba=2.0,2.0,16.0,2.0,2.0


In [21]:
# Compare top-ranked config for each model under each ranking method

comparison_rows = []

for model_name, df_model in df_by_model.items():
    if df_model.empty:
        continue

    df_ctd, df_prauc, df_combined = rank_per_model(df_model)

    top_configs = [
        ("Best Weighted CTD", df_ctd.iloc[0]),
        ("Best PR-AUC", df_prauc.iloc[0]),
        ("Best Combined Rank", df_combined.iloc[0]),
    ]

    for selection_method, row in top_configs:
        record = {
            "model_name": model_name,
            "selection_method": selection_method,
            "config_id": row["config_id"],
            "avg_weighted_ctd": row["avg_weighted_ctd"],
            "avg_toxic_pr_auc": row["avg_toxic_pr_auc"],
            "ctd_rank": row["ctd_rank"],
            "prauc_rank": row["prauc_rank"],
            "combined_rank_score": row["combined_rank_score"],
            "avg_rank": row["combined_rank_score"] / 2,
            "n_ctd": row["n_ctd"],
            "n_prauc": row["n_prauc"],
        }

        for col in [
            "trainable_params",
            "hidden_size",
            "depth_repr",
            "gru_layers",
            "mamba_layers",
            "mamba_d_state",
            "mamba_d_conv",
            "mamba_expand",
            "transformer_layers",
            "transformer_heads",
            "transformer_ff_ratio",
        ]:
            if col in row.index:
                record[col] = row[col]

        comparison_rows.append(record)

top_method_comparison_df = pd.DataFrame(comparison_rows)

display(
    top_method_comparison_df.sort_values(
        ["selection_method", "model_name"]
    ).reset_index(drop=True)
)

,model_name,selection_method,config_id,avg_weighted_ctd,avg_toxic_pr_auc,ctd_rank,prauc_rank,combined_rank_score,avg_rank,n_ctd,...,hidden_size,depth_repr,gru_layers,mamba_layers,mamba_d_state,mamba_d_conv,mamba_expand,transformer_layers,transformer_heads,transformer_ff_ratio
0,gru,Best Combined Rank,0ac6a244c89a,0.709324,0.481430,2.0,2.0,4.0,2.0,9,...,160,gru=2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,gru_transformer,Best Combined Rank,550f4e527d75,0.704851,0.481973,4.0,3.0,7.0,3.5,9,...,96,"gru=2.0,tr=1.0",2.0,NaN,NaN,NaN,NaN,1.0,2.0,2.0
2,mamba,Best Combined Rank,43dc32bbcf37,0.728337,0.483927,2.0,2.0,4.0,2.0,8,...,144,mamba=1.0,NaN,1.0,8.0,4.0,2.0,NaN,NaN,NaN
3,transformer,Best Combined Rank,1f3daa47e410,0.706293,0.474965,4.0,2.0,6.0,3.0,9,...,96,tr=2.0,NaN,NaN,NaN,NaN,NaN,2.0,4.0,2.0
4,gru,Best PR-AUC,8e60200ad5d3,0.668478,0.481648,7.0,1.0,8.0,4.0,9,...,96,gru=3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,gru_transformer,Best PR-AUC,a84515d2dc34,0.671507,0.486355,20.0,1.0,21.0,10.5,9,...,96,"gru=1.0,tr=1.0",1.0,NaN,NaN,NaN,NaN,1.0,2.0,4.0
6,mamba,Best PR-AUC,2c95fad373ce,0.687365,0.498257,45.0,1.0,46.0,23.0,8,...,144,mamba=2.0,NaN,2.0,32.0,2.0,2.0,NaN,NaN,NaN
7,transformer,Best PR-AUC,6d21350227a1,0.672923,0.490510,27.0,1.0,28.0,14.0,9,...,96,tr=4.0,NaN,NaN,NaN,NaN,NaN,4.0,2.0,3.0
8,gru,Best Weighted CTD,8bf3464c2bd8,0.734724,0.437933,1.0,6.0,7.0,3.5,9,...,160,gru=1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,gru_transformer,Best Weighted CTD,5727f292f129,0.724572,0.455911,1.0,29.0,30.0,15.0,9,...,128,"gru=1.0,tr=1.0",1.0,NaN,NaN,NaN,NaN,1.0,8.0,3.0


In [32]:
selected_cols = ['model_name','selection_method','config_id','avg_weighted_ctd','avg_toxic_pr_auc','ctd_rank','prauc_rank','avg_rank']

In [33]:
top_method_comparison_df[top_method_comparison_df['selection_method'] == 'Best Weighted CTD'].sort_values(['avg_weighted_ctd'], ascending=False).reset_index(drop=True)[selected_cols]

,model_name,selection_method,config_id,avg_weighted_ctd,avg_toxic_pr_auc,ctd_rank,prauc_rank,avg_rank
0,gru,Best Weighted CTD,8bf3464c2bd8,0.734724,0.437933,1.0,6.0,3.5
1,mamba,Best Weighted CTD,3adfed4cdef2,0.733845,0.452868,1.0,37.0,19.0
2,transformer,Best Weighted CTD,48bfdbd43034,0.724949,0.461530,1.0,13.0,7.0
3,gru_transformer,Best Weighted CTD,5727f292f129,0.724572,0.455911,1.0,29.0,15.0


In [34]:
top_method_comparison_df[top_method_comparison_df['selection_method'] == 'Best PR-AUC'].sort_values(['avg_weighted_ctd'], ascending=False).reset_index(drop=True)[selected_cols]

,model_name,selection_method,config_id,avg_weighted_ctd,avg_toxic_pr_auc,ctd_rank,prauc_rank,avg_rank
0,mamba,Best PR-AUC,2c95fad373ce,0.687365,0.498257,45.0,1.0,23.0
1,transformer,Best PR-AUC,6d21350227a1,0.672923,0.490510,27.0,1.0,14.0
2,gru_transformer,Best PR-AUC,a84515d2dc34,0.671507,0.486355,20.0,1.0,10.5
3,gru,Best PR-AUC,8e60200ad5d3,0.668478,0.481648,7.0,1.0,4.0


In [35]:
top_method_comparison_df[top_method_comparison_df['selection_method'] == 'Best Combined Rank'].sort_values(['avg_weighted_ctd'], ascending=False).reset_index(drop=True)[selected_cols]

,model_name,selection_method,config_id,avg_weighted_ctd,avg_toxic_pr_auc,ctd_rank,prauc_rank,avg_rank
0,mamba,Best Combined Rank,43dc32bbcf37,0.728337,0.483927,2.0,2.0,2.0
1,gru,Best Combined Rank,0ac6a244c89a,0.709324,0.481430,2.0,2.0,2.0
2,transformer,Best Combined Rank,1f3daa47e410,0.706293,0.474965,4.0,2.0,3.0
3,gru_transformer,Best Combined Rank,550f4e527d75,0.704851,0.481973,4.0,3.0,3.5
